# Online Graph Clustering (OGC) with GNNs

Graph Clustering on Reddit: Streaming and online graph clustering using dynamic neighborhood updates. This notebook implements the approach with `ClusterGCNConv` inside a `K3ClusteringNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `ClusterGCNConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid

title = "Online Graph Clustering (OGC) with GCN"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]

# 2. GCN Clustering Model
class K3ClusteringNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3ClusteringNet(dataset.num_features, 64, 7)
z = k3_model(data.x, data.edge_index)
print(f"Cluster embeddings computed: {z.shape}")

print("\n✓ K3-Node OGC execution completed successfully!")